# Member 4 — T2f (FLAIR) Pipeline
**Run cells top to bottom with Shift+Enter. Every cell has a visual check.**

| Cell | What it does |
|------|--------------|
| 1 | Setup — imports, seed, paths |
| 2 | Sanity check — load one patient, show all 4 modalities |
| 3 | Model check — confirm model builds and output shape is correct |
| 4 | Dataset check — confirm DataLoader returns the right shapes |
| 5 | Training |
| 6 | Evaluation — Dice / IoU / HD95 on test set |
| 7 | Grad-CAM XAI visualisation |

---
## Cell 1 — Setup

In [ ]:
import sys
from pathlib import Path

# Add repo root to path so shared.* imports work
REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from shared.seed import set_global_seed
set_global_seed()   # seed = 42

from shared.config import get_data_root, SPLITS_DIR, PATCH_SIZE
from shared.dataset import BraTSDataset, get_dataloader
from shared.metrics import compute_all_metrics

import torch
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

DATA_ROOT = get_data_root()
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Repo root : {REPO_ROOT}')
print(f'Data root : {DATA_ROOT}')
print(f'Device    : {DEVICE}')
print(f'Patch size: {PATCH_SIZE}')
print('Setup OK')

---
## Cell 2 — Sanity check: load one patient and visualise all 4 modalities

In [ ]:
from shared.preprocessing import preprocess_patient
import os

# Pick any patient from your subset
all_patients = sorted(os.listdir(DATA_ROOT))
PATIENT_ID   = all_patients[0]
PATIENT_DIR  = os.path.join(DATA_ROOT, PATIENT_ID)
print(f'Using patient: {PATIENT_ID}')

# Load all 4 modalities manually for display
import glob
def load_mod(mod):
    from shared.preprocessing import load_and_normalise
    path = glob.glob(os.path.join(PATIENT_DIR, f'*{mod}*.nii*'))[0]
    return load_and_normalise(path)

vols = {m: load_mod(m) for m in ['t1c', 't1n', 't2f', 't2w']}
seg_path = glob.glob(os.path.join(PATIENT_DIR, '*seg*.nii*'))[0]
seg = nib.load(seg_path).get_fdata()

# Best axial slice = most tumour voxels
tumour_per_slice = (seg > 0).sum(axis=(1, 2))
best_z = int(tumour_per_slice.argmax())
print(f'Best axial slice: z={best_z} ({tumour_per_slice[best_z]:.0f} tumour voxels)')

fig, axes = plt.subplots(1, 4, figsize=(18, 5), facecolor='#111')
titles = ['T1c (contrast)', 'T1n (native)', 'T2f FLAIR — YOUR modality', 'T2w']

for ax, (mod, title) in zip(axes, zip(['t1c','t1n','t2f','t2w'], titles)):
    ax.imshow(vols[mod][best_z].T, cmap='gray', origin='lower')
    # tumour core (label 2) = teal, enhancing tumour (label 4) = amber
    for lbl, col in [(2, '#5DCAA5'), (4, '#EF9F27')]:
        mask = (seg[best_z] == lbl)
        if mask.any():
            ax.contour(mask.T, levels=[0.5], colors=[col], linewidths=1.5)
    ax.set_title(title, color='white', fontsize=10)
    ax.axis('off')

from matplotlib.patches import Patch
legend = [Patch(color='#5DCAA5', label='Tumour core (label 2)'),
          Patch(color='#EF9F27', label='Enhancing tumour (label 4)')]
axes[-1].legend(handles=legend, loc='lower right',
                facecolor='#222', labelcolor='white', fontsize=8)
plt.suptitle(f'{PATIENT_ID} | axial z={best_z}', color='white', y=1.01)
plt.tight_layout()
plt.savefig('../results/figures/M4_sanity.png', dpi=120,
            bbox_inches='tight', facecolor='#111')
plt.show()
print('Notice: FLAIR (T2f) suppresses CSF — ventricles are dark, tumour is bright')

---
## Cell 3 — Model check

In [ ]:
from model import T2fSegModel

model = T2fSegModel(in_channels=1, base_ch=32).to(DEVICE)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)

patch = PATCH_SIZE[0]
dummy = torch.randn(1, 1, patch, patch, patch).to(DEVICE)
with torch.no_grad():
    out = model(dummy)

print(f'Input shape : {dummy.shape}')
print(f'Output shape: {out.shape}')   # should match input
print(f'Parameters  : {params:,}')
assert out.shape == dummy.shape, 'Shape mismatch!'
print('Model OK')

---
## Cell 4 — Dataset check

In [ ]:
train_ds = BraTSDataset(
    data_root  = DATA_ROOT,
    split_file = str(SPLITS_DIR / 'train_ids.txt'),
    modality   = 't2f',
    patch_size = PATCH_SIZE[0],
    augment    = False,
    patches_per_volume = 2,
)
print(f'Train dataset length: {len(train_ds)} patches')

sample = train_ds[0]
print(f'image shape : {sample["image"].shape}')  # (1, P, P, P)
print(f'label shape : {sample["label"].shape}')  # (P, P, P)
print(f'patient id  : {sample["patient_id"]}')
print(f'label unique values: {sample["label"].unique().tolist()}')

# Visualise one patch
img_np = sample['image'].squeeze().numpy()
lbl_np = sample['label'].numpy()
mid = img_np.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(10, 4), facecolor='#111')
axes[0].imshow(img_np[mid].T, cmap='gray', origin='lower')
axes[0].set_title('FLAIR patch', color='white')
axes[0].axis('off')
axes[1].imshow(img_np[mid].T, cmap='gray', origin='lower')
if lbl_np.sum() > 0:
    axes[1].contour(lbl_np[mid].T, levels=[0.5], colors=['#5DCAA5'], linewidths=1.5)
axes[1].set_title('Patch + tumour mask', color='white')
axes[1].axis('off')
plt.tight_layout()
plt.show()
print('Dataset OK')

---
## Cell 5 — Training
> This runs the full training loop. On CPU it will be slow — let it run in the background.
> Checkpoints are saved to `results/checkpoints/`.

In [ ]:
# Runs train.py as a subprocess so you can see output live in the notebook
import subprocess, sys
result = subprocess.run(
    [sys.executable, 'train.py'],
    cwd=str(Path().resolve()),   # run from member4_T2f/
)
print('Return code:', result.returncode)

---
## Cell 6 — Evaluation on test set

In [ ]:
from shared.config import CHECKPOINT_DIR
from shared.trainer import CheckpointManager
from shared.dataset import BraTSDataset, get_dataloader
from shared.metrics import compute_all_metrics
from model import T2fSegModel
import torch, numpy as np

model = T2fSegModel(in_channels=1, base_ch=32)
ckpt  = CheckpointManager(str(CHECKPOINT_DIR), 'member4_T2f')
model, _, epoch, val_dice = ckpt.load_best(model)
model = model.to(DEVICE).eval()
print(f'Loaded best checkpoint (epoch {epoch}, val Dice {val_dice:.4f})')

test_ds = BraTSDataset(
    data_root  = DATA_ROOT,
    split_file = str(SPLITS_DIR / 'test_ids.txt'),
    modality   = 't2f',
    patch_size = PATCH_SIZE[0],
    augment    = False,
    patches_per_volume = 2,
)
test_loader = get_dataloader(test_ds, batch_size=1, shuffle=False)

all_dice, all_iou, all_hd95 = [], [], []
with torch.no_grad():
    for batch in test_loader:
        images = batch['image'].to(DEVICE)
        labels = batch['label'].to(DEVICE)
        if labels.dim() == 4:
            labels = labels.unsqueeze(1)
        logits = model(images)
        m = compute_all_metrics(logits, labels)
        all_dice.append(m['dice'])
        all_iou.append(m['iou'])
        all_hd95.append(m['hd95'])

print(f'Test Dice : {np.nanmean(all_dice):.4f}')
print(f'Test IoU  : {np.nanmean(all_iou):.4f}')
print(f'Test HD95 : {np.nanmean(all_hd95):.1f} mm')

import pandas as pd
row = pd.DataFrame([{
    'Member': 'M4', 'Modality': 'T2f',
    'Architecture': 'ResUNet3D',
    'Dice_WT': round(np.nanmean(all_dice), 4),
    'IoU_WT':  round(np.nanmean(all_iou),  4),
    'HD95_WT': round(np.nanmean(all_hd95), 1),
}])
row.to_csv('../results/T2F/tables/M4_row.csv', index=False)
print('Saved to results/T2F/tables/M4_row.csv — share this with M5')
row

---
## Cell 7 — Grad-CAM XAI visualisation

In [ ]:
import torch.nn.functional as F
from shared.preprocessing import preprocess_patient
import matplotlib.pyplot as plt

# ── Grad-CAM hook ────────────────────────────────────────────────────────────
_feats, _grads = {}, {}

def fwd_hook(m, inp, out): _feats['f'] = out
def bwd_hook(m, gi, go):   _grads['g'] = go[0]

target_layer = model.enc4
h1 = target_layer.register_forward_hook(fwd_hook)
h2 = target_layer.register_full_backward_hook(bwd_hook)

# ── Load one test patient ─────────────────────────────────────────────────────
model.cpu().eval()
vol, seg = preprocess_patient(PATIENT_DIR, 't2f')
vol_t = torch.tensor(vol[np.newaxis]).float()  # (1,1,D,H,W)

# ── Forward + backward ───────────────────────────────────────────────────────
model.zero_grad()
pred = model(vol_t)
prob = torch.sigmoid(pred)
score = (prob * (prob > 0.5).float()).sum()
score.backward()

# ── Compute Grad-CAM ─────────────────────────────────────────────────────────
weights = _grads['g'].mean(dim=(2,3,4), keepdim=True)
cam = F.relu((weights * _feats['f']).sum(dim=1, keepdim=True))
cam = F.interpolate(cam, size=vol_t.shape[2:], mode='trilinear', align_corners=False)
cam_np = cam.squeeze().detach().numpy()
if cam_np.max() > 0:
    cam_np = (cam_np - cam_np.min()) / (cam_np.max() - cam_np.min())

h1.remove(); h2.remove()

# ── Visualise ─────────────────────────────────────────────────────────────────
vol_np = vol.squeeze()
seg_np = (seg > 0).astype(np.float32)
best_z = int(seg_np.sum(axis=(1,2)).argmax())
best_y = int(seg_np.sum(axis=(0,2)).argmax())
best_x = int(seg_np.sum(axis=(0,1)).argmax())

views = [
    ('Axial',    vol_np[best_z],       seg_np[best_z],       cam_np[best_z]),
    ('Coronal',  vol_np[:,best_y,:],   seg_np[:,best_y,:],   cam_np[:,best_y,:]),
    ('Sagittal', vol_np[:,:,best_x],   seg_np[:,:,best_x],   cam_np[:,:,best_x]),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9), facecolor='#111')
for col, (name, mri, gt, cam) in enumerate(views):
    axes[0, col].imshow(mri.T, cmap='gray', origin='lower')
    if gt.sum() > 0:
        axes[0, col].contour(gt.T, levels=[0.5], colors=['#5DCAA5'], linewidths=1.5)
    axes[0, col].set_title(f'FLAIR — {name}', color='white')
    axes[0, col].axis('off')

    axes[1, col].imshow(mri.T, cmap='gray', origin='lower')
    axes[1, col].imshow(cam.T, cmap='hot', alpha=0.55, origin='lower', vmin=0, vmax=1)
    if gt.sum() > 0:
        axes[1, col].contour(gt.T, levels=[0.5], colors=['#5DCAA5'], linewidths=1.5)
    axes[1, col].set_title(f'Grad-CAM — {name}', color='white')
    axes[1, col].axis('off')

axes[0,0].set_ylabel('Raw FLAIR', color='white')
axes[1,0].set_ylabel('Grad-CAM overlay', color='white')
plt.suptitle(f'M4 XAI | {PATIENT_ID}', color='white', fontsize=13)
plt.tight_layout()
plt.savefig(f'../results/T2F/figures/M4_gradcam_{PATIENT_ID}.png', dpi=130,
            bbox_inches='tight', facecolor='#111')
plt.show()
print('Check: does the hot heatmap sit over the green tumour contour?')
print('If yes — Grad-CAM is working correctly for FLAIR')